In [7]:
import torch


In [8]:
print(f"Is GPU available? {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU ❌. Check your PyTorch installation.")

Is GPU available? True
GPU Name: NVIDIA GeForce GTX 1650


In [3]:
from ultralytics import YOLO
 
model = YOLO('yolov8n.pt')

results = model.train(data=r'C:\Users\dev88\Downloads\archive (1)\data.yaml', 
    epochs=30, 
    imgsz=320,           
    batch=8,          
    workers=6,   
    half=True,
    model='yolov8n.pt' )

print("Training finished!")

Ultralytics 8.3.233  Python-3.11.7 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\dev88\Downloads\archive (1)\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=True, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train7, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspe

In [ ]:
import cv2
from ultralytics import YOLO
import math
from collections import deque


model_path = r"C:\Users\dev88\Untitled Folder\Garbage dectection system\runs\detect\train7\weights\best.pt"

model = YOLO(model_path)


cap = cv2.VideoCapture(0)
cap.set(3, 640)
cap.set(4, 480)

FRAME_CENTER_X = 320
FRAME_CENTER_Y = 240
SMOOTH_WINDOW = 5
DEAD_ZONE = 30 


box_queue = deque(maxlen=SMOOTH_WINDOW)

def avg_box(box_list):
    """Averages coordinates to reduce jitter."""
    if not box_list: return None
    n = len(box_list)
    sx1 = sum(b[0] for b in box_list) / n
    sy1 = sum(b[1] for b in box_list) / n
    sx2 = sum(b[2] for b in box_list) / n
    sy2 = sum(b[3] for b in box_list) / n
    return int(sx1), int(sy1), int(sx2), int(sy2)

print("Tracking Closest Target... Press 'q' to exit.")

def get_position():
    while True:
        success, img = cap.read()
        if not success: break

    
        results = model(img, stream=True, conf=0.5, verbose=False)

        best_target = None
        min_dist = float('inf')

   
        for r in results:
            for box in r.boxes:
         
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                
               
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2
                
                
                dist = math.sqrt((cx - FRAME_CENTER_X)**2 + (cy - FRAME_CENTER_Y)**2)
                
                
                if dist < min_dist:
                    min_dist = dist
                    best_target = (x1, y1, x2, y2)

        
        if best_target:
           
            box_queue.append(best_target)
            
            
            s_box = avg_box(list(box_queue))
            if s_box:
                sx1, sy1, sx2, sy2 = s_box
                
                
                center_x = (sx1 + sx2) // 2
                center_y = (sy1 + sy2) // 2
                final_dist = math.sqrt((center_x - FRAME_CENTER_X)**2 + (center_y - FRAME_CENTER_Y)**2)

                
                if final_dist < DEAD_ZONE:
                    direction = "CENTER"
                    color = (0, 255, 0) 
                elif center_x < FRAME_CENTER_X:
                    direction = "LEFT"
                    color = (0, 255, 255) 
                else:
                    direction = "RIGHT"
                    color = (0, 255, 255) 

             
                cv2.circle(img, (center_x, center_y), 6, (0, 0, 255), -1)
                
               
                cv2.line(img, (FRAME_CENTER_X, FRAME_CENTER_Y), (center_x, center_y), color, 2)
                
                # 3. Display Values
                label = f"{direction} | Dist: {int(final_dist)}"
                cv2.putText(img, label, (center_x - 40, center_y - 20), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)


        else:
            if len(box_queue) > 0:
                box_queue.popleft()

       
        cv2.drawMarker(img, (FRAME_CENTER_X, FRAME_CENTER_Y), (200, 200, 200), 
                       markerType=cv2.MARKER_CROSS, markerSize=20, thickness=1)

        cv2.imshow("Precision Garbage Tracker", img)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    get_position()

Tracking Closest Target... Press 'q' to exit.


In [ ]:
model.export(format = 'onnx' , device=0)

In [2]:
import torch

# URL for the pre-trained YOLOv8 Garbage Model (Medium Size)
model_url = "https://huggingface.co/turhancan97/yolov8-segment-trash-detection/resolve/main/yolov8m-seg.pt"

print("Downloading pre-trained Garbage Detection Model...")
torch.hub.download_url_to_file(model_url, 'garbage_seg_best.pt')
print("Download Complete! Saved as 'garbage_seg_best.pt'")

100%|██████████████████████████████████████████████████████████████████████████████| 52.2M/52.2M [01:14<00:00, 738kB/s]

Download Complete! Saved as 'garbage_seg_best.pt'


In [10]:
import cv2
from ultralytics import YOLO
import math
import numpy as np
from collections import deque

# --- CONFIGURATION ---
model_path = r"garbage_seg_best.pt"
model = YOLO(model_path)

# Camera Setup
cap = cv2.VideoCapture(0)
cap.set(3, 640)
cap.set(4, 480)

# Constants
FRAME_CENTER_X = 320
FRAME_CENTER_Y = 240
DEAD_ZONE_X = 50       # Tolerance for "CENTER"
LOCK_THRESHOLD = 20    # Frames to lock
LOST_THRESHOLD = 10    # Frames to wait before resetting

# --- STABILIZER ---
class Stabilizer:
    def __init__(self, window_size=5):
        self.q = deque(maxlen=window_size)
    
    def update(self, val):
        self.q.append(val)
        return sum(self.q) / len(self.q)

    def reset(self):
        self.q.clear()

# --- TARGET MANAGER ---
class TargetManager:
    def __init__(self):
        self.state = "SCANNING"
        self.locked_id = None
        self.lock_counter = 0    
        self.lost_counter = 0    
        self.smooth_x = Stabilizer(window_size=8)
        self.smooth_y = Stabilizer(window_size=8)

    def process(self, detections):
        best_target = None

        # 1. If LOCKED, only look for locked_id
        if self.state == "LOCKED":
            found = False
            for det in detections:
                if det['id'] == self.locked_id:
                    best_target = det
                    found = True
                    self.lost_counter = 0
                    break
            if not found:
                self.lost_counter += 1
                if self.lost_counter > LOST_THRESHOLD:
                    self.reset_lock()

        # 2. If VERIFYING, check if target persists
        elif self.state == "VERIFYING":
            found = False
            for det in detections:
                if det['id'] == self.locked_id:
                    best_target = det
                    found = True
                    self.lock_counter += 1
                    break
            if found:
                if self.lock_counter >= LOCK_THRESHOLD:
                    self.state = "LOCKED"
            else:
                self.reset_lock()

        # 3. If SCANNING, pick closest to center
        if self.state == "SCANNING" and detections:
            closest = min(detections, key=lambda x: x['dist_from_center'])
            self.locked_id = closest['id']
            self.state = "VERIFYING"
            self.lock_counter = 1
            return self.get_smoothed_result(closest)

        if best_target:
            return self.get_smoothed_result(best_target)
        
        return None

    def get_smoothed_result(self, target):
        x1, y1, x2, y2 = target['box']
        raw_cx = (x1 + x2) // 2
        raw_cy = (y1 + y2) // 2

        smooth_cx = int(self.smooth_x.update(raw_cx))
        smooth_cy = int(self.smooth_y.update(raw_cy))
        
        return {
            'id': target['id'],
            'center': (smooth_cx, smooth_cy),
            'state': self.state
        }

    def reset_lock(self):
        self.state = "SCANNING"
        self.locked_id = None
        self.lock_counter = 0
        self.lost_counter = 0
        self.smooth_x.reset()
        self.smooth_y.reset()

tracker = TargetManager()

def get_position():
    while True:
        success, img = cap.read()
        if not success: break
        
        # 1. TRACKING
        results = model.track(img, persist=True, verbose=False, tracker="bytetrack.yaml")
        current_detections = []
        
        for r in results:
            if r.boxes.id is None: continue
            
            boxes = r.boxes.xyxy.cpu().numpy().astype(int)
            ids = r.boxes.id.cpu().numpy().astype(int)
            
            for box, track_id in zip(boxes, ids):
                x1, y1, x2, y2 = box
                cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
                dist_error = math.sqrt((cx - FRAME_CENTER_X)**2 + (cy - FRAME_CENTER_Y)**2)
                
                current_detections.append({
                    'id': track_id,
                    'box': box,
                    'dist_from_center': dist_error
                })
        
        # 2. PROCESS TRACKER
        target = tracker.process(current_detections)
        
        # 3. CLEAN DRAWING (No Box, No ID)
        if target:
            cx, cy = target['center']
            
            # Distance Calculation
            offset_x = cx - FRAME_CENTER_X
            offset_y = cy - FRAME_CENTER_Y
            center_distance = math.sqrt(offset_x**2 + offset_y**2)
            
            # Direction
            direction = "CENTER"
            color = (0, 255, 0)
            
            if offset_x < -DEAD_ZONE_X:
                direction = "<< LEFT"
                color = (0, 255, 255)
            elif offset_x > DEAD_ZONE_X:
                direction = "RIGHT >>"
                color = (0, 255, 255)
                
            # --- VISUALIZATION ---
            # 1. Draw Red Point (Target Center)
            cv2.circle(img, (cx, cy), 8, (0, 0, 255), -1) 
            
            # 2. Draw Tether Line (White line from Crosshair to Target)
            cv2.line(img, (FRAME_CENTER_X, FRAME_CENTER_Y), (cx, cy), (255, 255, 255), 1)
            
            # 3. Text: Distance & Direction ONLY
            label = f"{int(center_distance)} px | {direction}"
            
            # Put text slightly above the point
            cv2.putText(img, label, (cx - 50, cy - 20), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        # Draw Crosshair
        cv2.drawMarker(img, (FRAME_CENTER_X, FRAME_CENTER_Y), (200, 200, 200), 
                       markerType=cv2.MARKER_CROSS, markerSize=30, thickness=2)

        cv2.imshow("Garbage Point Tracker", img)
        if cv2.waitKey(1) & 0xFF == ord('q'): break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    get_position()